# Semantic Query Agent

Natural-language-to-SQL agent using the OpenAI Agents SDK through Abacus.AI RouteLLM and the local Microsoft Northwind SQLite database.

In [ ]:
from __future__ import annotations

import os
import re
import sqlite3
import time
from pathlib import Path

import pandas as pd
from agents import Agent, Runner, set_default_openai_api, set_default_openai_client, set_tracing_disabled
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'data' / 'northwind.db').exists():
    ROOT_DIR = ROOT_DIR.parent
DATABASE_PATH = ROOT_DIR / 'data' / 'northwind.db'
load_dotenv(ROOT_DIR / '.env')
api_key = os.getenv('ABACUS_API_KEY')
if not api_key:
    raise RuntimeError('Set ABACUS_API_KEY in .env before using the agent.')
set_default_openai_client(AsyncOpenAI(base_url=os.getenv('ABACUS_BASE_URL', 'https://routellm.abacus.ai/v1'), api_key=api_key), use_for_tracing=False)
set_default_openai_api('chat_completions')
set_tracing_disabled(True)


In [ ]:
def database_schema() -> str:
    with sqlite3.connect(DATABASE_PATH) as connection:
        rows = connection.execute("SELECT name, sql FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall()
    return '\n\n'.join(f'{name}: {sql}' for name, sql in rows)

SCHEMA_CONTEXT = database_schema()

class SQLPlan(BaseModel):
    sql: str = Field(description='One read-only SQLite SELECT or WITH query.')
    explanation: str = Field(description='Short business-language explanation of what the query answers.')
    schema_context: str = Field(description='Relevant tables and columns used for the query.')

SYSTEM_PROMPT = '''You are Semantic Query Agent. Translate a business question into one safe SQLite query for Microsoft Northwind. Return structured output. Generate exactly one read-only SELECT or WITH query. Never generate INSERT, UPDATE, DELETE, DROP, ALTER, ATTACH, DETACH, PRAGMA, or multiple statements. Quote Northwind identifiers containing spaces, including \"Order Details\". Use only this schema:\n\n''' + SCHEMA_CONTEXT
agent = Agent(name='Semantic Query Agent', model=os.getenv('ABACUS_MODEL', 'route-llm'), instructions=SYSTEM_PROMPT, output_type=SQLPlan)


In [ ]:
FORBIDDEN_SQL = re.compile(r'\b(INSERT|UPDATE|DELETE|DROP|ALTER|ATTACH|DETACH|PRAGMA|VACUUM|REINDEX|CREATE|REPLACE)\b', re.IGNORECASE)

def validate_sql(sql: str) -> str:
    statement = sql.strip().rstrip(';').strip()
    if not statement.upper().startswith(('SELECT', 'WITH')) or ';' in statement or FORBIDDEN_SQL.search(statement):
        raise ValueError('The agent did not return a single read-only SQLite query.')
    return statement

def answer_question(question: str) -> dict[str, object]:
    plan = Runner.run_sync(agent, question).final_output
    if not isinstance(plan, SQLPlan):
        raise RuntimeError('The agent did not return a SQL plan.')
    sql = validate_sql(plan.sql)
    started = time.perf_counter()
    with sqlite3.connect(f'file:{DATABASE_PATH}?mode=ro', uri=True) as connection:
        dataframe = pd.read_sql_query(sql, connection)
    execution_ms = (time.perf_counter() - started) * 1000
    row_summary = f'Returned {len(dataframe):,} row' + ('' if len(dataframe) == 1 else 's') + '.'
    return {'response': f'{plan.explanation} {row_summary}', 'sql': sql, 'schema_context': plan.schema_context, 'execution_ms': execution_ms, 'dataframe': dataframe}
